# TF-IDF Text Feature Extraction (IEMOCAP)

Extract TF-IDF features once and export to CSV for reuse in downstream tests/models.

This notebook is tuned for high-core CPU machines and will use many physical cores for transcript parsing and numerical kernels.

In [ ]:
from __future__ import annotations

import os
import re
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import importlib.util

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('Could not locate repository root (missing pyproject.toml).')

def detect_physical_cores() -> int:
    # Linux-first physical core detection using /proc/cpuinfo.
    cpuinfo = Path('/proc/cpuinfo')
    if cpuinfo.exists():
        pairs: set[tuple[str, str]] = set()
        physical_id = None
        core_id = None
        with cpuinfo.open('r', encoding='utf-8', errors='ignore') as handle:
            for raw in handle:
                line = raw.strip()
                if not line:
                    if physical_id is not None and core_id is not None:
                        pairs.add((physical_id, core_id))
                    physical_id = None
                    core_id = None
                    continue
                if line.startswith('physical id'):
                    physical_id = line.split(':', 1)[1].strip()
                elif line.startswith('core id'):
                    core_id = line.split(':', 1)[1].strip()
        if physical_id is not None and core_id is not None:
            pairs.add((physical_id, core_id))
        if pairs:
            return len(pairs)

    # Fallback for other environments.
    logical = os.cpu_count() or 1
    return max(1, logical // 2)

REPO_ROOT = find_repo_root(Path.cwd())
META_CSV = REPO_ROOT / 'datasets' / 'IEMOCAP' / 'iemocap_full_dataset.csv'
IEMOCAP_ROOT = REPO_ROOT / 'datasets' / 'IEMOCAP'
OUT_DIR = REPO_ROOT / 'extracted_features' / 'text'
OUT_CSV = OUT_DIR / 'tfidf_features.csv'
VOCAB_CSV = OUT_DIR / 'tfidf_vocabulary.csv'

OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_PHYSICAL_CORES = detect_physical_cores()
RESERVED_CORES = 2
CPU_WORKERS = max(1, NUM_PHYSICAL_CORES - RESERVED_CORES)

# Set thread controls before importing numpy/scikit-learn heavy modules.
os.environ['OMP_NUM_THREADS'] = str(CPU_WORKERS)
os.environ['MKL_NUM_THREADS'] = str(CPU_WORKERS)
os.environ['OPENBLAS_NUM_THREADS'] = str(CPU_WORKERS)
os.environ['NUMEXPR_NUM_THREADS'] = str(CPU_WORKERS)

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

HAS_TORCH = importlib.util.find_spec('torch') is not None
CUDA_AVAILABLE = False
if HAS_TORCH:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()

HAS_RAPIDS_TFIDF = (
    importlib.util.find_spec('cudf') is not None
    and importlib.util.find_spec('cuml') is not None
)
USE_CUDA_TFIDF = CUDA_AVAILABLE and HAS_RAPIDS_TFIDF

print(f'Physical cores detected: {NUM_PHYSICAL_CORES}')
print(f'CPU workers for this notebook: {CPU_WORKERS}')
print(f'CUDA available: {CUDA_AVAILABLE}')
print(f'RAPIDS TF-IDF available: {HAS_RAPIDS_TFIDF}')
if CUDA_AVAILABLE and not HAS_RAPIDS_TFIDF:
    print('CUDA is available, but RAPIDS TF-IDF is not installed. Using CPU path (recommended for this environment).')

META_CSV, OUT_CSV


In [ ]:
LINE_RE = re.compile(r'^(?P<utt>\S+)\s+\[[^\]]+\]:\s*(?P<text>.*)$')

def parse_transcript_file(txt_path: str) -> dict[str, str]:
    file_index: dict[str, str] = {}
    with Path(txt_path).open('r', encoding='utf-8', errors='ignore') as handle:
        for line in handle:
            m = LINE_RE.match(line.strip())
            if not m:
                continue
            utt = m.group('utt')
            text = m.group('text').strip()
            if not text:
                continue
            if utt in file_index:
                file_index[utt] = (file_index[utt] + ' ' + text).strip()
            else:
                file_index[utt] = text
    return file_index

def build_transcript_index(iemocap_dir: Path, max_workers: int) -> dict[str, str]:
    files = sorted(iemocap_dir.glob('Session*/dialog/transcriptions/*.txt'))
    if not files:
        raise FileNotFoundError('No transcript files found under Session*/dialog/transcriptions/*.txt')

    use_workers = max(1, min(max_workers, len(files)))
    print(f'Parsing {len(files)} transcript files with {use_workers} workers...')
    start = time.perf_counter()

    partial_indexes: list[dict[str, str]] = []
    if use_workers == 1:
        for path in files:
            partial_indexes.append(parse_transcript_file(str(path)))
    else:
        with ProcessPoolExecutor(max_workers=use_workers) as ex:
            for partial in ex.map(parse_transcript_file, [str(p) for p in files], chunksize=4):
                partial_indexes.append(partial)

    index: dict[str, str] = {}
    for partial in partial_indexes:
        for utt, text in partial.items():
            if utt in index:
                index[utt] = (index[utt] + ' ' + text).strip()
            else:
                index[utt] = text

    elapsed = time.perf_counter() - start
    print(f'Transcript indexing complete: {len(index)} utterances ({elapsed:.2f}s)')
    return index

df = pd.read_csv(META_CSV)
df['emotion'] = df['emotion'].astype(str).str.strip().str.lower()
df = df[(df['emotion'] != 'xxx') & (df['agreement'] > 0)].copy()
df['utt_id'] = df['path'].apply(lambda p: Path(p).stem)

transcripts = build_transcript_index(IEMOCAP_ROOT, max_workers=CPU_WORKERS)
df['text'] = df['utt_id'].map(transcripts)
df_text = df[df['text'].notna() & (df['text'].str.len() > 0)].copy()

print(f'Rows with text: {len(df_text)}')
df_text.shape


In [ ]:
train_mask = df_text['session'].isin([1, 2, 3, 4])

start = time.perf_counter()

# TF-IDF extraction remains CPU-oriented in this environment.
if USE_CUDA_TFIDF:
    from cuml.feature_extraction.text import TfidfVectorizer as CuTfidfVectorizer
    import cudf

    vectorizer = CuTfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000,
    )

    train_text_gpu = cudf.Series(df_text.loc[train_mask, 'text'].astype(str).tolist())
    all_text_gpu = cudf.Series(df_text['text'].astype(str).tolist())

    vectorizer.fit(train_text_gpu)
    matrix_gpu = vectorizer.transform(all_text_gpu)
    matrix = matrix_gpu.get() if hasattr(matrix_gpu, 'get') else matrix_gpu

    # Feature names / IDF compatibility differs across RAPIDS versions.
    terms = vectorizer.get_feature_names() if hasattr(vectorizer, 'get_feature_names') else vectorizer.get_feature_names_out()
    idf_values = vectorizer.idf_ if hasattr(vectorizer, 'idf_') else [None] * len(terms)
else:
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000,
        dtype=np.float32,
    )

    vectorizer.fit(df_text.loc[train_mask, 'text'])
    matrix = vectorizer.transform(df_text['text'])
    terms = vectorizer.get_feature_names_out()
    idf_values = vectorizer.idf_

vectorize_elapsed = time.perf_counter() - start
print(f'TF-IDF complete in {vectorize_elapsed:.2f}s | shape={matrix.shape}')

feature_cols = [f'tfidf_{idx:05d}' for idx in range(matrix.shape[1])]
tfidf_df = pd.DataFrame.sparse.from_spmatrix(matrix, columns=feature_cols)

out = df_text[['path', 'session', 'method', 'gender', 'emotion', 'n_annotators', 'agreement', 'utt_id', 'text']].reset_index(drop=True)
out['split'] = out['session'].apply(lambda value: 'train' if value in {1, 2, 3, 4} else 'test')
out = pd.concat([out, tfidf_df], axis=1)

vocab = pd.DataFrame({
    'feature_col': feature_cols,
    'term': terms,
    'idf': idf_values,
})

write_start = time.perf_counter()
out.to_csv(OUT_CSV, index=False)
vocab.to_csv(VOCAB_CSV, index=False)
write_elapsed = time.perf_counter() - write_start

print(f'Saved TF-IDF feature CSV: {OUT_CSV}')
print(f'Saved TF-IDF vocabulary CSV: {VOCAB_CSV}')
print(f'Rows: {len(out)} | TF-IDF columns: {len(vocab)}')
print(f'CSV write time: {write_elapsed:.2f}s')
out.shape
